In [3]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
# ============================================================
# STAGE 1: SETUP for H2A
# ============================================================
!pip install -q transformers datasets accelerate sentencepiece sacrebleu evaluate peft
!git clone https://github.com/rfuentesfe/EgyptianHieroglyphicText




# Step 1: Download Franken dataset
!python /kaggle/working/EgyptianHieroglyphicText/scripts/1_download_franken_dataset.py

# Step 2: Merge both datasets
!python /kaggle/working/EgyptianHieroglyphicText/scripts/3_merge_datasets.py

# Step 3: Adjust image sizes to 224x224
!python /kaggle/working/EgyptianHieroglyphicText/scripts/4_adjust_files.py

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.5.1+cu121 requires torch==2.5.1, but you have torch 2.12.1 which is incompatible.
torchvision 0.20.1+cu121 requires torch==2.5.1, but you have torch 2.12.1 which is incompatible.
Cloning into 'EgyptianHieroglyphicText'...
remote: Enumerating objects: 20067, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (22/22), done.
^Cceiving objects:  87% (17459/20067), 428.33 MiB | 41.42 MiB/s
python3: can't open file '/kaggle/working/EgyptianHieroglyphicText/scripts/1_download_franken_dataset.py': [Errno 2] No such file or directory
python3: can't open file '/kaggle/working/EgyptianHieroglyphicText/scripts/3_merge_datasets.py': [Errno 2] No such file or directory
python3: can't open file '/kaggle/working/EgyptianHieroglyphicText/scripts/4_adjust_files.py': [Errno 2] No such 

In [4]:
import os
import shutil

franken = '/kaggle/working/GlyphFranken2025'
hiero   = '/kaggle/working/EgyptianHieroglyphicText/dataset'
merged  = '/kaggle/working/Glyph2025'

os.makedirs(merged, exist_ok=True)

def merge_into(src, dest):
    for cls in os.listdir(src):
        src_cls  = os.path.join(src, cls)
        dest_cls = os.path.join(dest, cls.lower())
        os.makedirs(dest_cls, exist_ok=True)
        for f in os.listdir(src_cls):
            src_file  = os.path.join(src_cls, f)
            dest_file = os.path.join(dest_cls, f.lower())
            if not os.path.exists(dest_file):
                shutil.copy2(src_file, dest_file)

print("Merging Franken...")
merge_into(franken, merged)
print("Merging Hiero...")
merge_into(hiero, merged)

# Verify
classes = os.listdir(merged)
total = sum(len(os.listdir(os.path.join(merged, c))) for c in classes)
print(f"\nClasses: {len(classes)}")
print(f"Total images: {total}")

Merging Franken...


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/GlyphFranken2025'

In [ ]:
!python /kaggle/working/EgyptianHieroglyphicText/scripts/4_adjust_files.py

In [ ]:
import os
from PIL import Image

merged = '/kaggle/working/Glyph2025'
classes = os.listdir(merged)
total = sum(len(os.listdir(os.path.join(merged, c))) for c in classes)
print(f"Classes: {len(classes)}")
print(f"Total images: {total}")

# Check a sample image size
sample_class = classes[0]
sample_img = os.listdir(os.path.join(merged, sample_class))[0]
img = Image.open(os.path.join(merged, sample_class, sample_img))
print(f"Sample image size: {img.size}")

In [ ]:
import os, cv2, json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, datasets, models
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from PIL import Image
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

DATASET_PATH = '/kaggle/working/Glyph2025'
IMG_H, IMG_W = 300, 300
BATCH_SIZE   = 32
SEED         = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
print('Setup done.')

In [ ]:
def enhance_contrast(image: Image.Image) -> Image.Image:
    """
    CLAHE contrast enhancement.
    Works on LAB color space to preserve colors while fixing brightness.
    Better than simple histogram equalization.
    """
    img_np = np.clip(np.array(image), 0, 255).astype('uint8')
    lab    = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe  = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    cl     = clahe.apply(l)
    limg   = cv2.merge((cl, a, b))
    result = cv2.cvtColor(limg, cv2.COLOR_LAB2RGB)
    return Image.fromarray(result)


# ── Training transforms (strong augmentation) ───────────────
train_transforms = transforms.Compose([
    transforms.Resize((IMG_H, IMG_W)),
    transforms.Lambda(enhance_contrast), # Keeps baseline image quality uniform

    # 1. Advanced Augment (Handles color, brightness, and basic shifts elegantly)
    transforms.RandomAutocontrast(p=0.3),
    transforms.RandomAdjustSharpness(sharpness_factor=2, p=0.3),

    # 2. Safe Geometric Adjustments (No horizontal flipping!)
    transforms.RandomRotation(degrees=15),
    transforms.RandomPerspective(distortion_scale=0.15, p=0.2),

    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),

    # 3. Patch Erasing (Prevents model from over-relying on a single stroke)
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.08), value='random'),
])

# ── Validation transforms (no augmentation) ─────────────────
val_transforms = transforms.Compose([
    transforms.Resize((IMG_H, IMG_W)),
    transforms.Lambda(enhance_contrast),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

inference_transforms = val_transforms 

print('Transforms defined.')

In [ ]:
class SubsetWithTransform(Dataset):
    """Applies a different transform to a subset of a dataset."""
    def __init__(self, dataset, indices, transform):
        self.dataset   = dataset
        self.indices   = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        image, label = self.dataset[self.indices[idx]]
        if self.transform:
            image = self.transform(image)
        return image, label


full_dataset = datasets.ImageFolder(root=DATASET_PATH, transform=None)
class_names  = full_dataset.classes
num_classes  = len(class_names)
print(f'Classes     : {num_classes}')
print(f'Total images: {len(full_dataset)}')
print(f'Avg per class: {len(full_dataset)/num_classes:.1f}')

n_val   = int(0.2 * len(full_dataset))
n_train = len(full_dataset) - n_val

train_indices, val_indices = torch.utils.data.random_split(
    range(len(full_dataset)), [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED)
)

train_dataset = SubsetWithTransform(full_dataset, train_indices.indices, train_transforms)
val_dataset   = SubsetWithTransform(full_dataset, val_indices.indices,   val_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

# Save class mapping for inference / NLP pipeline
with open('class_mapping.json', 'w', encoding='utf-8') as f:
    json.dump(full_dataset.class_to_idx, f, ensure_ascii=False, indent=2)

idx_to_class = {v: k for k, v in full_dataset.class_to_idx.items()}

print(f'\nclass_mapping.json saved')
print(f'Train: {n_train} | Val: {n_val}')

In [ ]:
def build_model(num_classes: int, freeze_backbone: bool = True) -> nn.Module:
    """
    efficientnet_v2_s with improved classifier head:
    - Higher dropout (0.5 / 0.3)
    - BatchNorm1d after first linear for stable training
    """
    model = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1)
    in_features = model.classifier[1].in_features
        
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False
                
    # Replace the final linear layer with our custom deep head
    model.classifier[1] = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),          # smoother than ReLU, consistent with modern backbones
            nn.Dropout(p=0.5),  # high dropout during frozen phase
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(p=0.3),  # lower before final layer
            nn.Linear(256, num_classes)
        )
    
    return model # Added to prevent the NoneType crash!


# ── Run and verify parameters ───────────────────────────
model     = build_model(num_classes, freeze_backbone=True).to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())

print(f'Trainable params : {trainable:,}')
print(f'Total params     : {total:,}')

In [ ]:
def mixup_data(x, y, alpha=0.2):
    """Apply MixUp to a batch. Returns mixed images and both label sets."""
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0)).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[idx]
    return mixed_x, y, y[idx], lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    """MixUp-aware cross entropy loss."""
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


print('MixUp helpers ready.')

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)


def run_epoch(model, loader, optimizer, criterion, train=True, use_mixup=False):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0.0, 0

    with torch.set_grad_enabled(train):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            if train and use_mixup:
                images, y_a, y_b, lam = mixup_data(images, labels, alpha=0.2)
                outputs = model(images)
                loss    = mixup_criterion(criterion, outputs, y_a, y_b, lam)
                preds   = outputs.argmax(1)
                correct += (lam * (preds == y_a).float() +
                           (1 - lam) * (preds == y_b).float()).sum().item()
            else:
                outputs = model(images)
                loss    = criterion(outputs, labels)
                correct += (outputs.argmax(1) == labels).sum().item()

            if train:
                optimizer.zero_grad()
                loss.backward()
                # Gradient clipping — prevents exploding gradients
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            total_loss += loss.item()
            total      += labels.size(0)

    return total_loss / len(loader), correct / total


print('Training loop defined.')

In [ ]:
EPOCHS_P1  = 100
PATIENCE_P1 = 10

optimizer_p1 = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3,
    weight_decay=1e-3
)
scheduler_p1 = CosineAnnealingLR(optimizer_p1, T_max=EPOCHS_P1, eta_min=1e-5)

history_p1   = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_p1  = 0.0
no_improve   = 0

print('── Phase 1: Head only (backbone frozen) ──')
for epoch in range(EPOCHS_P1):
    train_loss, train_acc = run_epoch(model, train_loader, optimizer_p1, criterion,
                                      train=True,  use_mixup=True)
    val_loss,   val_acc   = run_epoch(model, val_loader,   optimizer_p1, criterion,
                                      train=False, use_mixup=False)
    scheduler_p1.step()

    history_p1['train_loss'].append(train_loss)
    history_p1['train_acc'].append(train_acc)
    history_p1['val_loss'].append(val_loss)
    history_p1['val_acc'].append(val_acc)

    print(f'Ep {epoch+1:03d}/{EPOCHS_P1} | '
          f'Loss {train_loss:.3f} | '
          f'Train {train_acc:.3f} | '
          f'Val {val_acc:.3f}', end='')

    if val_acc > best_val_p1:
        best_val_p1 = val_acc
        no_improve  = 0
        torch.save(model.state_dict(), 'best_efficientNets_p1.pth')
        print(' ✅')
    else:
        no_improve += 1
        print(f' (no improve {no_improve}/{PATIENCE_P1})')
        if no_improve >= PATIENCE_P1:
            print(f'\n⏹ Early stopping at epoch {epoch+1}')
            break

print(f'\nPhase 1 best val accuracy: {best_val_p1:.3f}')

In [ ]:
from torch.optim.lr_scheduler import (CosineAnnealingWarmRestarts,
                                       LinearLR, SequentialLR)

# ── 1. Load checkpoint ───────────────────────────────────────
ckpt = torch.load('best_efficientNets_p1.pth', map_location=device, weights_only=False)
model.load_state_dict(ckpt)
best_val_p1 = best_val_p1 if 'best_val_p1' in vars() else 0.0
print(f'Loaded Phase 1 best checkpoint (val={best_val_p1:.3f})')

# ── 2. Initial unfreeze (last 3 blocks + classifier) ────────
for param in model.parameters():
    param.requires_grad = False
for block in list(model.features.children())[-3:]:
    for param in block.parameters():
        param.requires_grad = True
for param in model.classifier.parameters():
    param.requires_grad = True
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable (initial): {trainable:,} / {total:,}')

# ── 3. Hyperparameters ───────────────────────────────────────
EPOCHS_P2           = 100
PATIENCE_P2         = 15
UNFREEZE_HALF_EPOCH = 30
UNFREEZE_ALL_EPOCH  = 50
WARMUP_EPOCHS       = 5

# ── 4. Optimizer ─────────────────────────────────────────────
optimizer_p2 = AdamW([
    {'params': model.features.parameters(),   'lr': 1e-5},
    {'params': model.classifier.parameters(), 'lr': 1e-4},
], weight_decay=1e-4)

# ── 5. Scheduler (warmup → CAWR) ─────────────────────────────
warmup       = LinearLR(optimizer_p2, start_factor=0.1, end_factor=1.0, total_iters=WARMUP_EPOCHS)
cosine       = CosineAnnealingWarmRestarts(optimizer_p2, T_0=25, T_mult=1, eta_min=1e-6)
scheduler_p2 = SequentialLR(optimizer_p2, schedulers=[warmup, cosine], milestones=[WARMUP_EPOCHS])

# Restart points: warmup ends at 5, then every 25 → {5, 30, 55, 80}
restart_epochs = {WARMUP_EPOCHS + i * 25 for i in range(4)}

# ── 6. Training state ────────────────────────────────────────
history_p2     = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_p2    = 0.0
no_improve     = 0
half_unfrozen  = False
fully_unfrozen = False

# ── 7. Training loop ─────────────────────────────────────────
print('\n── Phase 2 : 100-epoch full fine-tune ──')
print(f'Restart epochs : {sorted(restart_epochs)}')
print(f'Unfreeze 50%   : epoch {UNFREEZE_HALF_EPOCH}')
print(f'Unfreeze 100%  : epoch {UNFREEZE_ALL_EPOCH}')

for epoch in range(EPOCHS_P2):

    # ── Gradual unfreeze ─────────────────────────────────────
    if epoch == UNFREEZE_HALF_EPOCH and not half_unfrozen:
        blocks = list(model.features.children())
        mid    = len(blocks) // 2
        for block in blocks[mid:]:
            for param in block.parameters():
                param.requires_grad = True
        for pg in optimizer_p2.param_groups:
            pg['weight_decay'] = 5e-4       # tighten regularization
        half_unfrozen = True
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f'\n── 50% unfreeze at epoch {epoch+1} | Trainable: {trainable:,} ──')

    if epoch == UNFREEZE_ALL_EPOCH and not fully_unfrozen:
        for param in model.parameters():
            param.requires_grad = True
        fully_unfrozen = True
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f'\n── Full unfreeze at epoch {epoch+1} | Trainable: {trainable:,} ──')

    if epoch == 75:
        for pg in optimizer_p2.param_groups:
            pg['weight_decay'] = 1e-4       # relax in final convergence phase
        print(f'\n── Weight decay relaxed at epoch {epoch+1} ──')

    # ── Train / validate ─────────────────────────────────────
    train_loss, train_acc = run_epoch(model, train_loader, optimizer_p2, criterion, train=True,  use_mixup=True)
    val_loss,   val_acc   = run_epoch(model, val_loader,   None,         criterion, train=False, use_mixup=False)

    # Step the SequentialLR manager
    scheduler_p2.step()

    # ── Reset patience at CAWR restarts ──────────────────────
    if epoch in restart_epochs:
        no_improve = 0

    # ── History tracking ─────────────────────────────────────
    history_p2['train_loss'].append(train_loss)
    history_p2['train_acc'].append(train_acc)
    history_p2['val_loss'].append(val_loss)
    history_p2['val_acc'].append(val_acc)

    current_lr = optimizer_p2.param_groups[0]['lr']
    print(f'Ep {epoch+1:03d}/{EPOCHS_P2} | '
          f'Loss {train_loss:.3f} | '
          f'Train {train_acc:.3f} | '
          f'Val {val_acc:.3f} | '
          f'LR {current_lr:.2e}', end='')

    # ── Single Weights Checkpoint ────────────────────────────
    if val_acc > best_val_p2:
        best_val_p2 = val_acc
        no_improve  = 0
        # Saves only the primary overall best weights
        torch.save(model.state_dict(), 'final_hieroglyph_weights.pth')
        print(f' ✅ Best weights saved (val={val_acc:.3f})')
    else:
        no_improve += 1
        print(f' (no improve {no_improve}/{PATIENCE_P2})')
        if no_improve >= PATIENCE_P2:
            print(f'\n⏹ Early stopping at epoch {epoch+1}')
            break

print(f'\nPhase 2 Complete! Best validation accuracy achieved: {best_val_p2:.3f}')

In [ ]:
def plot_history(h1, h2):
    train_acc  = h1['train_acc']  + h2['train_acc']
    val_acc    = h1['val_acc']    + h2['val_acc']
    train_loss = h1['train_loss'] + h2['train_loss']
    val_loss   = h1['val_loss']   + h2['val_loss']
    p1_end     = len(h1['train_acc'])
    epochs     = range(1, len(train_acc) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Accuracy plot
    axes[0].plot(epochs, train_acc, label='Train Acc', color='royalblue', linewidth=2)
    axes[0].plot(epochs, val_acc,   label='Val Acc',   color='tomato',    linewidth=2)
    axes[0].axvline(x=p1_end, color='gray', linestyle='--', linewidth=1.5, label='Phase 1 → 2')
    axes[0].set_title('Accuracy', fontsize=13, fontweight='bold')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # Loss plot
    axes[1].plot(epochs, train_loss, label='Train Loss', color='royalblue', linewidth=2)
    axes[1].plot(epochs, val_loss,   label='Val Loss',   color='tomato',    linewidth=2)
    axes[1].axvline(x=p1_end, color='gray', linestyle='--', linewidth=1.5, label='Phase 1 → 2')
    axes[1].set_title('Loss', fontsize=13, fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.suptitle(
        f'Training History  |  Best Val: {max(val_acc):.3f}',
        fontsize=14, fontweight='bold'
    )
    plt.tight_layout()
    plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: training_history.png')


plot_history(history_p1, history_p2)